In [1]:
import pandas as pd
import sqlite3

In [2]:
df_val = pd.read_csv("/Users/kiara/Documents/matkassen-etl-pipeline/matkassen-etl/notebook/matkassen_validation.csv")
df_val.shape

(424, 21)

In [3]:
import os
os.getcwd()

'/Users/kiara/Documents/matkassen-etl-pipeline/matkassen-etl/notebook'

In [4]:
os.listdir()

['pipeline',
 'matkassen_data.csv',
 '.DS_Store',
 'matkassen_tvattad.ipynb',
 'etl_pipeline.ipynb',
 'matkassen_validation.csv',
 'sqlite_load.ipynb',
 'matkassen.db',
 'kpi_features.ipynb',
 'matkassen_data_clean.csv',
 'validation_sqlite.ipynb']

In [5]:
import pandas as pd

df_val = pd.read_csv("matkassen_validation.csv")
df_val.shape

(424, 21)

In [3]:
import sys
sys.path.append("/Users/kiara/Documents/matkassen-etl-pipeline/matkassen-etl/notebook/pipeline")

# Kopiera run_pipeline funktionen här istället för import
import pandas as pd
from pathlib import Path
from transformers import pipeline

def clean_week_price(df: pd.DataFrame) -> pd.DataFrame:
    """
    Städar veckapris: klarar '499 kr', '499SEK', '499:-', komma/punkt m.m.
    """
    s = df["veckapris"].astype(str).str.lower()
    s = s.str.replace(",", ".", regex=False)
    s = (
        s.str.replace("sek", "", regex=False)
         .str.replace("kr", "", regex=False)
         .str.replace(":-", "", regex=False)
         .str.replace(" ", "", regex=False)
    )
    s = s.str.extract(r"(\d+(?:\.\d+)?)", expand=False)
    df["veckapris"] = pd.to_numeric(s, errors="coerce")
    return df

def normalize_categories(df: pd.DataFrame) -> pd.DataFrame:
    """
    Normaliserar kasstyp så samma kategori inte finns i flera varianter
    """
    df["kasstyp"] = df["kasstyp"].str.strip()

    df["kasstyp"] = df["kasstyp"].replace({
        # Klassisk
        "Classic": "Klassisk",
        "KLASSISK": "Klassisk",
        "klassisk": "Klassisk",
        "Standard": "Klassisk",

        # Familj
        "Family": "Familj",
        "FAMILJ": "Familj",
        "familj": "Familj",
        "Familjekassen": "Familj",

        # Vegetarisk
        "Vegetarian": "Vegetarisk",
        "VEGETARIAN": "Vegetarisk",
        "VEGETARISK": "Vegetarisk",
        "vegetarisk": "Vegetarisk",
        "Veg": "Vegetarisk",
        "veg": "Vegetarisk",
        "Veggie": "Vegetarisk",

        # Snabb & Enkel
        "Snabb & enkel": "Snabb & Enkel",
        "snabb": "Snabb & Enkel",
        "30-min": "Snabb & Enkel",
        "Express": "Snabb & Enkel",
        "Quick": "Snabb & Enkel",
    })
    return df

def clean_delivery_status(df: pd.DataFrame) -> pd.DataFrame:
    """
    Normaliserar leveransstatus till Levererad / Missad
    """
    df["leveransstatus"] = df["leveransstatus"].str.strip()

    df["leveransstatus"] = df["leveransstatus"].replace({
        "Delivered": "Levererad",
        "LEVERERAD": "Levererad",
        "levererad": "Levererad",
        "Ok": "Levererad",
        "OK": "Levererad",
        "missad": "Missad",
        "MISSAD": "Missad",
        "Missed": "Missad",
        "Ej levererad": "Missad",
    })
    return df

def parse_dates(df: pd.DataFrame) -> pd.DataFrame:
    """
    Konverterar datumkolumner till datetime
    """
    date_cols = [
        "pren_startdatum",
        "paus_från",
        "paus_till",
        "pren_avslutsdatum",
        "leveransdatum",
        "omdömesdatum"
    ]
    for col in date_cols:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors="coerce")
    return df

def add_feature_engineering(df: pd.DataFrame) -> pd.DataFrame:
    """
    Skapar nya kolumner för analys
    """
    if "leveransdatum" in df.columns:
        df["weekday"] = df["leveransdatum"].dt.day_name()
        df["month"] = df["leveransdatum"].dt.month
        df["year"] = df["leveransdatum"].dt.year
    
    # Försäljning per vecka (om veckopris finns)
    if "veckapris" in df.columns:
        df["sales_per_week"] = df["veckapris"]
    
    # Ledtider (om pren_startdatum och leveransdatum finns)
    if "pren_startdatum" in df.columns and "leveransdatum" in df.columns:
        df["lead_time_days"] = (df["leveransdatum"] - df["pren_startdatum"]).dt.days
    
    return df

def add_sentiment_analysis(df: pd.DataFrame) -> pd.DataFrame:
    """
    Lägger till sentimentanalys med BERT om omdöme_text finns
    """
    if "omdöme_text" not in df.columns:
        print("Ingen omdöme_text kolumn, hoppar över sentimentanalys")
        return df
    
    df_sentiment = df[df["omdöme_text"].notna()].copy()
    if len(df_sentiment) == 0:
        print("Inga omdömen att analysera")
        return df
    
    print(f"Analyserar sentiment för {len(df_sentiment)} omdömen...")
    
    # Ladda BERT-modell
    sentiment_pipeline = pipeline("sentiment-analysis", model="nlptown/bert-base-multilingual-uncased-sentiment")
    
    sentiments = []
    texter = df_sentiment["omdöme_text"].astype(str).tolist()
    
    for text in texter:
        try:
            result = sentiment_pipeline(text)[0]
            label = result['label']
            # Mappa till Positiv/Neutral/Negativ
            if label in ['4 stars', '5 stars']:
                sentiment = 'Positiv'
            elif label == '3 stars':
                sentiment = 'Neutral'
            else:
                sentiment = 'Negativ'
            sentiments.append(sentiment)
        except Exception as e:
            print(f"Fel vid analys: {e}")
            sentiments.append('Neutral')
    
    df_sentiment["sentiment"] = sentiments
    df = df.merge(df_sentiment[["leverans_id", "sentiment"]], on="leverans_id", how="left")
    print("Sentimentanalys klar")
    return df

def run_pipeline(df: pd.DataFrame) -> pd.DataFrame:
    """
    Huvudfunktion som kör hela ETL-pipelinen
    """
    df = clean_week_price(df)
    df = normalize_categories(df)
    df = clean_delivery_status(df)
    df = parse_dates(df)
    df = add_feature_engineering(df)
    df = add_sentiment_analysis(df)
    return df

/Users/kiara/Library/Python/3.10/lib/python/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
df_val_clean = run_pipeline(df_val)
df_val_clean.shape

Analyserar sentiment för 162 omdömen...


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 11051.24it/s]


Sentimentanalys klar


(424, 27)

In [5]:
import sqlite3

conn = sqlite3.connect("/Users/kiara/Documents/matkassen-etl-pipeline/matkassen-etl/notebook/matkassen.db")

df_val_clean.to_sql(
    "matkassen_validation",
    conn,
    if_exists="replace",
    index=False
)

conn.close()
print("Valideringsdata laddad till SQLite-tabellen 'matkassen_validation'")

Valideringsdata laddad till SQLite-tabellen 'matkassen_validation'


In [6]:
import sqlite3

conn = sqlite3.connect("/Users/kiara/Documents/matkassen-etl-pipeline/matkassen-etl/notebook/matkassen.db")

df_val_clean.to_sql(
    "matkassen_validation",
    conn,
    if_exists="replace",
    index=False
)

conn.close()
print("Valideringsdata laddad till SQLite-tabellen 'matkassen_validation'")

# Verifiera laddning
conn = sqlite3.connect("/Users/kiara/Documents/matkassen-etl-pipeline/matkassen-etl/notebook/matkassen.db")
df_check = pd.read_sql("SELECT COUNT(*) as count FROM matkassen_validation", conn)
conn.close()
print(f"Verifiering: {df_check.iloc[0,0]} rader i tabellen 'matkassen_validation'")

Valideringsdata laddad till SQLite-tabellen 'matkassen_validation'
Verifiering: 424 rader i tabellen 'matkassen_validation'
